# Pipeline de análisis de percepción ciudadana — Población víctima de Quibdó

**Proyecto:** Sistema de análisis de percepción ciudadana de la población víctima
**Organización:** Oficina de Enlace de Víctimas — Alcaldía Municipal de Quibdó (Chocó)
**Módulo:** Gerencia de Proyectos y Analítica · Especialización en Analítica de Datos · UNAULA

Arquitectura **Medallion** (Bronze → Silver → Gold) sobre Databricks, con un modelo de
clasificación (Random Forest) registrado en **MLflow**.

> **Datos sintéticos:** el CSV de entrada fue generado por algoritmo con fines académicos.
> No corresponde a registros reales ni a personas identificables. El tratamiento de datos
> reales exigiría cumplimiento de la Ley 1581 de 2012 (Habeas Data).


## 0. Parámetros

Ajusta `catalog`, `schema` y `volume_path` para que coincidan con tu workspace de Databricks.


In [ ]:
# Parámetros del entorno (ajústalos a tu workspace)
catalog     = "workspace"
schema      = "victimas_quibdo"
volume_path = f"/Volumes/{catalog}/{schema}/landing"   # sube aquí el CSV
csv_file    = "PQRSD_PERCEPCION_VICTIMAS_QUIBDO.csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
print("Catálogo:", catalog, "| Esquema:", schema)


## 1. Capa Bronze — carga cruda

Se carga el CSV tal cual, sin transformaciones, para conservar el origen y garantizar
trazabilidad completa (linaje).


In [ ]:
from pyspark.sql import functions as F

bronze = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(f"{volume_path}/{csv_file}"))

(bronze.write.mode("overwrite")
    .saveAsTable(f"{catalog}.{schema}.bronze_percepcion_victimas"))

print("Registros en Bronze:", bronze.count())
bronze.show(5, truncate=60)


## 2. Capa Silver — limpieza y enriquecimiento

Procesos de calidad:
- Normalización de tipos (`Dias_respuesta` numérico, `Anio` entero).
- Estandarización de texto (recorte de espacios).
- Bandera de sentimiento negativo (`es_negativo`) derivada de la clasificación.
- Deduplicación por folio y validación de rangos lógicos (días de respuesta > 0).


In [ ]:
silver = (spark.table(f"{catalog}.{schema}.bronze_percepcion_victimas")
    .withColumn("Dias_respuesta", F.col("Dias_respuesta").cast("int"))
    .withColumn("Anio", F.col("Anio").cast("int"))
    .withColumn("Zona", F.trim(F.col("Zona")))
    .withColumn("Categoria", F.trim(F.col("Categoria")))
    .withColumn("Sentimiento", F.trim(F.col("Sentimiento")))
    .withColumn("es_negativo", (F.col("Sentimiento") == "Negativo").cast("int"))
    .dropDuplicates(["Folio"])
    .filter(F.col("Dias_respuesta") > 0))

(silver.write.mode("overwrite")
    .saveAsTable(f"{catalog}.{schema}.silver_percepcion_limpio"))

print("Registros en Silver:", silver.count())
silver.select("Folio","Zona","Categoria","Sentimiento","es_negativo","Dias_respuesta").show(5, truncate=40)


## 3. Capa Gold — agregados para negocio

Tabla `gold_percepcion_zonas`: una fila por combinación **zona–categoría–período**, con
volumen de PQRSD, porcentaje de sentimiento negativo, tiempo promedio de respuesta, y un
**nivel de alerta** de insatisfacción para priorizar la atención.


In [ ]:
gold = (spark.table(f"{catalog}.{schema}.silver_percepcion_limpio")
    .groupBy("Periodo", "Anio", "Zona", "Tipo_zona", "Categoria")
    .agg(
        F.count("*").alias("total_pqrsd"),
        F.sum("es_negativo").alias("pqrsd_negativas"),
        F.round(F.avg("Dias_respuesta"), 1).alias("dias_promedio"),
    )
    .withColumn("pct_negativo", F.round(F.col("pqrsd_negativas") / F.col("total_pqrsd") * 100, 1))
    .withColumn("nivel_alerta",
        F.when(F.col("pct_negativo") >= 55, "Extrema")
         .when(F.col("pct_negativo") >= 45, "Alta")
         .when(F.col("pct_negativo") >= 32, "Media")
         .otherwise("Moderada"))
    .withColumn("alerta_alta", (F.col("pct_negativo") >= 45).cast("int")))

(gold.write.mode("overwrite")
    .saveAsTable(f"{catalog}.{schema}.gold_percepcion_zonas"))

print("Filas en Gold:", gold.count())
gold.orderBy(F.desc("pct_negativo")).show(10, truncate=30)


## 4. Análisis descriptivo

Exploración de la tabla Gold: temas predominantes, insatisfacción por zona y evolución
temporal.


In [ ]:
import pandas as pd
g = spark.table(f"{catalog}.{schema}.gold_percepcion_zonas").toPandas()

print("== Insatisfacción promedio por categoría ==")
print(g.groupby("Categoria")["pct_negativo"].mean().round(1).sort_values(ascending=False))

print("\n== Zonas con mayor insatisfacción ==")
print(g.groupby("Zona")["pct_negativo"].mean().round(1).sort_values(ascending=False).head(5))

print("\n== Evolución del % negativo por período ==")
print(g.groupby("Periodo")["pct_negativo"].mean().round(1))


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

cat = g.groupby("Categoria")["pct_negativo"].mean().sort_values()
ax[0].barh(cat.index, cat.values, color="#0F6B3F")
ax[0].set_title("Insatisfacción promedio por tema (%)")
ax[0].set_xlabel("% negativo")

evo = g.groupby("Periodo")["pct_negativo"].mean()
ax[1].plot(evo.index, evo.values, marker="o", color="#C0392B", linewidth=2.5)
ax[1].set_title("Evolución del sentimiento negativo")
ax[1].set_ylabel("% negativo")
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Modelo de Machine Learning — scoring de alerta

Se entrena un **Random Forest** sobre la tabla Gold para estimar la probabilidad de que una
combinación **zona–categoría–período** entre en **alerta alta o extrema** de insatisfacción,
usando como variables el tema, el tipo de zona, el volumen de PQRSD y el tiempo de respuesta.

El experimento se registra en **MLflow** (parámetros, métricas y modelo serializado).


In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = spark.table(f"{catalog}.{schema}.gold_percepcion_zonas").toPandas()

features_cat = ["Categoria", "Tipo_zona"]
features_num = ["total_pqrsd", "dias_promedio"]
X = df[features_cat + features_num]
y = df["alerta_alta"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), features_cat),
    ("num", "passthrough", features_num),
])
clf = Pipeline([
    ("pre", pre),
    ("rf", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)),
])

with mlflow.start_run(run_name="rf_alerta_insatisfaccion"):
    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]
    pred = clf.predict(X_test)

    auc = roc_auc_score(y_test, proba)
    f1  = f1_score(y_test, pred)

    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metric("AUC_ROC", auc)
    mlflow.log_metric("F1_score", f1)
    mlflow.sklearn.log_model(clf, "modelo_alerta")

    print(f"AUC-ROC : {auc:.3f}")
    print(f"F1-score: {f1:.3f}")
    print()
    print(classification_report(y_test, pred, target_names=["Sin alerta alta", "Alerta alta"]))


## 6. Consumo — tabla Gold para el tablero

La tabla `gold_percepcion_zonas` es el punto único de consumo para el tablero de monitoreo
(Power BI, Databricks SQL o la app web del proyecto). Expuesta vía **Databricks SQL Warehouse**,
alimenta las tarjetas KPI, el ranking de zonas y el semáforo de alerta.


In [ ]:
# Vista de priorización: zonas-tema en alerta alta o extrema, ordenadas por insatisfacción
(spark.table(f"{catalog}.{schema}.gold_percepcion_zonas")
    .filter(F.col("nivel_alerta").isin("Alta", "Extrema"))
    .orderBy(F.desc("pct_negativo"))
    .select("Periodo","Zona","Categoria","total_pqrsd","pct_negativo","dias_promedio","nivel_alerta")
    .show(15, truncate=30))


---
### Resumen del flujo

| Capa | Tabla generada | Contenido |
|---|---|---|
| Bronze | `bronze_percepcion_victimas` | CSV crudo, sin transformar |
| Silver | `silver_percepcion_limpio` | Tipado, limpieza, deduplicación, bandera de sentimiento |
| Gold | `gold_percepcion_zonas` | Agregados por zona-tema-período + nivel de alerta |
| ML | Modelo en MLflow | Random Forest de scoring de alerta de insatisfacción |

**Datos sintéticos, uso académico. El tratamiento de datos reales exige cumplimiento de la Ley 1581 de 2012.**
